# Approaches to find candidate genes (genomic approach)

In [1]:
# open vcf file
import numpy as np
import cyvcf2
import pandas as pd

SAMPLE    = "SBC10"
VCF_PATH  = "../results/sift4g/SBC10.private.sift4g.vcf.gz"
GENE_INFO = "../resources/NCBI_FTP/gene_info_4558"

def load_sorbi_to_loc(gene_info_path: str) -> dict:
    gi = pd.read_csv(gene_info_path, sep="\t", usecols=["Symbol", "LocusTag"])
    return gi.set_index("LocusTag")["Symbol"].to_dict()

In [2]:
sorbi_to_loc = load_sorbi_to_loc(GENE_INFO)

In [8]:
ANN_FIELDS = [
    "ann_allele", "ann_effect", "ann_impact", "ann_gene_name", "ann_gene_id",
    "ann_feature_type", "ann_feature_id", "ann_biotype", "ann_rank",
    "ann_hgvs_c", "ann_hgvs_p", "ann_cdna_pos", "ann_cds_pos", "ann_aa_pos",
    "ann_distance", "ann_extra",
]

SIFT_FIELDS = [
    "sift_allele", "sift_transcript", "sift_gene_id", "sift_gene_name",
    "sift_region", "sift_variant_type", "sift_aa_change", "sift_aa_pos",
    "sift_score", "sift_median", "sift_num_seqs", "sift_allele_type",
    "sift_prediction",
]

# SIFT4G writes DELETERIOUS (score < 0.05) or TOLERATED; NA means non-coding
_SIFT_PRIORITY = {"DELETERIOUS": 0, "TOLERATED": 1}


def _parse_ann(raw: str) -> list[dict]:
    records = []
    for entry in raw.split(","):
        parts = entry.split("|")
        parts += [""] * (len(ANN_FIELDS) - len(parts))
        records.append(dict(zip(ANN_FIELDS, parts[:len(ANN_FIELDS)])))
    return records


def _parse_siftinfo(raw: str) -> dict:
    """Returns allele -> best SIFT record.

    ANN transcript IDs (EER*, KXG*, OQU* — JGI/Phytozome) differ from SIFT
    transcript IDs (XM_* — NCBI RefSeq), so the only shared key per VCF line
    is the alt allele sequence. When multiple SIFT records exist for the same
    allele (different transcripts), keep the most damaging one.
    """
    by_allele: dict[str, dict] = {}
    for entry in raw.split(","):
        parts = entry.split("|")
        parts += [""] * (len(SIFT_FIELDS) - len(parts))
        d = dict(zip(SIFT_FIELDS, parts[:len(SIFT_FIELDS)]))
        allele = d["sift_allele"]
        pred = d.get("sift_prediction", "")
        current = by_allele.get(allele)
        if current is None or _SIFT_PRIORITY.get(pred, 99) < _SIFT_PRIORITY.get(current.get("sift_prediction", ""), 99):
            by_allele[allele] = d
    return by_allele


def parse_vcf(vcf_path: str, sorbi_to_loc: dict) -> pd.DataFrame:
    """
    Parse a SIFT4G+SnpEff annotated VCF into a DataFrame.
    One row per ANN annotation entry; SIFT columns joined by allele.
    gene_symbol is derived from ann_gene_id via sorbi_to_loc.
    """
    vcf = cyvcf2.VCF(vcf_path)
    rows = []
    for variant in vcf:
        gt_arr = variant.genotypes[0]
        sep = "|" if gt_arr[2] else "/"
        base = {
            "chrom":  variant.CHROM,
            "pos":    variant.POS,
            "ref":    variant.REF,
            "alt":    ",".join(variant.ALT),
            "qual":   variant.QUAL,
            "filter": variant.FILTER,
            "gt":     f"{gt_arr[0]}{sep}{gt_arr[1]}",
        }

        ann_raw  = variant.INFO.get("ANN")
        sift_raw = variant.INFO.get("SIFTINFO")
        annotations    = _parse_ann(ann_raw)       if ann_raw  else [{}]
        sift_by_allele = _parse_siftinfo(sift_raw) if sift_raw else {}

        for ann in annotations:
            sift = sift_by_allele.get(ann.get("ann_allele", ""), {})
            row  = {**base, **ann, **sift}
            row["gene_symbol"] = sorbi_to_loc.get(ann.get("ann_gene_id", ""), "")
            rows.append(row)

    df = pd.DataFrame(rows)
    df["sift_score"] = pd.to_numeric(df["sift_score"], errors="coerce")
    df["sift_score_c"] = 1 - df["sift_score"]
    return df

# def scoring_annotated_variants(df):
    

In [9]:
df = parse_vcf(VCF_PATH, sorbi_to_loc)
df

,chrom,pos,ref,alt,qual,filter,gt,ann_allele,ann_effect,ann_impact,...,sift_region,sift_variant_type,sift_aa_change,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c
0,NC_012870.2,1619,G,A,54.540001,None,1|0,A,upstream_gene_variant,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NC_012870.2,1619,G,A,54.540001,None,1|0,A,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NC_012870.2,5784,A,"AGG,AG",25.700001,None,1/2,AG,downstream_gene_variant,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NC_012870.2,5784,A,"AGG,AG",25.700001,None,1/2,AGG,downstream_gene_variant,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NC_012870.2,5784,A,"AGG,AG",25.700001,None,1/2,AG,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899795,NC_012879.2,61233582,GTT,G,35.779999,None,1/1,G,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
899796,NC_012879.2,61233615,A,T,29.270000,None,1/1,T,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
899797,NC_012879.2,61233619,T,G,27.490000,None,1/1,G,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
899798,NC_012879.2,61233644,A,AG,35.650002,None,1/1,AG,intergenic_region,MODIFIER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
"stop_gained" in df["ann_effect"].unique()

True

In [13]:
# some problem: mismatch between ANN annotated gene id and SIFT annotated
df[df["sift_gene_id"] == "LOC8060857"]

,chrom,pos,ref,alt,qual,filter,gt,ann_allele,ann_effect,ann_impact,...,sift_region,sift_variant_type,sift_aa_change,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c
1476,NC_012870.2,819679,T,TA,20.080000,None,1/1,TA,3_prime_UTR_variant,MODIFIER,...,UTR_3,FRAMESHIFT INSERTION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1477,NC_012870.2,819679,T,TA,20.080000,None,1/1,TA,upstream_gene_variant,MODIFIER,...,UTR_3,FRAMESHIFT INSERTION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1478,NC_012870.2,819679,T,TA,20.080000,None,1/1,TA,downstream_gene_variant,MODIFIER,...,UTR_3,FRAMESHIFT INSERTION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1479,NC_012870.2,819679,T,TA,20.080000,None,1/1,TA,downstream_gene_variant,MODIFIER,...,UTR_3,FRAMESHIFT INSERTION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1480,NC_012870.2,819679,T,TA,20.080000,None,1/1,TA,downstream_gene_variant,MODIFIER,...,UTR_3,FRAMESHIFT INSERTION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1576,NC_012870.2,821566,GCTT,G,41.209999,None,1/1,G,5_prime_UTR_variant,MODIFIER,...,UTR_5,NONFRAMESHIFT DELETION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1577,NC_012870.2,821566,GCTT,G,41.209999,None,1/1,G,upstream_gene_variant,MODIFIER,...,UTR_5,NONFRAMESHIFT DELETION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1578,NC_012870.2,821566,GCTT,G,41.209999,None,1/1,G,downstream_gene_variant,MODIFIER,...,UTR_5,NONFRAMESHIFT DELETION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN
1579,NC_012870.2,821566,GCTT,G,41.209999,None,1/1,G,downstream_gene_variant,MODIFIER,...,UTR_5,NONFRAMESHIFT DELETION,NA/NA,NA,NaN,NA,NA,NA,NA,NaN


In [10]:
# distribution of c sift score
# I need the number to score non-HIGH non-SIFT-scores-having annotated variant sites
print(pd.to_numeric(df["sift_score_c"], errors='coerce').describe())
print("\n")
for impact in df["ann_impact"].unique():
    print(f"{impact}\n{pd.to_numeric(df[df['ann_impact'] == impact]['sift_score_c'], errors='coerce').describe()}\n")

count    29792.000000
mean         0.405319
std          0.406767
min          0.000000
25%          0.000000
50%          0.320000
75%          0.840000
max          1.000000
Name: sift_score_c, dtype: float64


MODIFIER
count    16198.000000
mean         0.417114
std          0.410778
min          0.000000
25%          0.000000
50%          0.350000
75%          0.870000
max          1.000000
Name: sift_score_c, dtype: float64

MODERATE
count    7139.000000
mean        0.619101
std         0.362045
min         0.000000
25%         0.350000
50%         0.740000
75%         0.950000
max         1.000000
Name: sift_score_c, dtype: float64

HIGH
count    34.000000
mean      0.843824
std       0.325651
min       0.000000
25%       0.922500
50%       1.000000
75%       1.000000
max       1.000000
Name: sift_score_c, dtype: float64

LOW
count    6421.000000
mean        0.135554
std         0.266123
min         0.000000
25%         0.000000
50%         0.000000
75%         0.080000
max      

In [19]:
for impact in df["ann_impact"].unique():
    print(f"{impact}", len(df["ann_impact"].isna()))

MODIFIER 899800
MODERATE 899800
HIGH 899800
LOW 899800


0         False
1         False
2         False
3         False
4         False
          ...  
899795    False
899796    False
899797    False
899798    False
899799    False
Name: ann_impact, Length: 899800, dtype: bool

In [6]:
# maximum approach (aka worst variant)


In [7]:
# sum approach